## Crossreactivity Data and Table Creation

In [1]:
# ── Environment setup ─────────────────────────────────────────────────────
# ProxiPal is the project's custom analysis library (ProxiPal.py in repo root).

import sys
from pathlib import Path

module_dir = Path.cwd().parent
sys.path.append(str(module_dir))

from ProxiPal import *
# ProxiPal also imports pandas, numpy, scipy, matplotlib — these are
# available after this line without separate import statements.
# See: https://github.com/jeffsmith8/NfL-iPLA-paper


OS:				 Windows-11-10.0.26200-SP0
Python version:		 3.12.6 | packaged by conda-forge | (main, Sep 22 2024, 14:01:26) [MSC v.1941 64 bit (AMD64)]
Date and Time:		 2026-05-26 10:00:53
cwd:		 C:\Users\smith.j\OneDrive - wehi.edu.au\Documents\CFHAC_ProxiPal\ProxiPal\user_documents\NfL-PLA Development Paper\NfL-PLA Development Paper Figure Data\GitHub\NfL-iPLA-paper\iPLA_crossreactivity
base_path:		 C:\Users\smith.j\OneDrive - wehi.edu.au\Documents\CFHAC_ProxiPal\ProxiPal
python_folder:		 C:\Users\smith.j\OneDrive - wehi.edu.au\Documents\CFHAC_ProxiPal\ProxiPal\python
templates_folder:	 C:\Users\smith.j\OneDrive - wehi.edu.au\Documents\CFHAC_ProxiPal\ProxiPal\templates
data_folder:		 C:\Users\smith.j\OneDrive - wehi.edu.au\Documents\CFHAC_ProxiPal\ProxiPal\data
samples_folder:		 C:\Users\smith.j\OneDrive - wehi.edu.au\Documents\CFHAC_ProxiPal\ProxiPal\samples
quality_folder:		 C:\Users\smith.j\OneDrive - wehi.edu.au\Documents\CFHAC_ProxiPal\ProxiPal\quality
user_downloads:		 C:\Users\smith.

In [4]:
xreact_single = pd.read_csv("iPLA_xreact_single.csv")
xreact_single.head(1)

,Unnamed: 0,well,position,target,sample_id,tube_id,rep_id,dilution,var_1,var_2,...,sybr (eds; multicomponent data),fam (eds; multicomponent data),cy5 (eds; multicomponent data),well (eds; melt curve raw data),well position (eds; melt curve raw data),reading (eds; melt curve raw data),temperature (eds; melt curve raw data),fluorescence (eds; melt curve raw data),derivative (eds; melt curve raw data),expt_folder_csv
0,4898,3,A3,NfL,NfH(200000)_progen#62210,NaN,2,1.0,1,0,...,"[146779.344, 149957.5, 152673.109, 153402.922,...",NaN,NaN,3.0,A3,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[60.303, 60.795, 61.287, 61.779, 62.271, 62.76...","[1.901, 1.869, 1.848, 1.82, 1.801, 1.772, 1.74...","[0.05, 0.05, 0.051, 0.051, 0.051, 0.051, 0.05,...",240514_e65_kruti


### Estimated pg/ml with single targets: NfH, NfM, GFAP, b-Tubulin

In [6]:
# Trim to mean values
xreact_single_means = xreact_single[['sample_id', 'expt_folder_csv', 'rep_id','SLR; log10(x); exc_std0; ct; mean_ng/L']].drop_duplicates(['expt_folder_csv', 'rep_id'])

import re

# Load the means file
# df_means = pd.read_csv("xreact_expt_lanes3to4_means.csv")

# Identify correct column
value_col = 'SLR; log10(x); exc_std0; ct; mean_ng/L'

# Extract target (before '('), fix naming, and dose (inside '()')
xreact_single_means['target'] = xreact_single_means['sample_id'].apply(
    lambda x: "b-Tubulin" if x.startswith("bTubulin") else x.split("(")[0].strip()
)
xreact_single_means['dose'] = xreact_single_means['sample_id'].apply(
    lambda x: float(re.search(r"\(([^)]+)\)", x).group(1)) if re.search(r"\(([^)]+)\)", x) else np.nan
)

# Group by target and dose to compute mean
df_stats = (
    xreact_single_means.groupby(['dose', 'target'])[value_col]
    .mean()
    .reset_index()
)

# Pivot into a table: rows = dose, columns = target
df_result = df_stats.pivot(index='dose', columns='target', values=value_col)

# Format to 2 decimal places
df_result = df_result.round(2)

# Displayor export
df_result.to_csv('CrossReactivity_single.csv')
df_result


target,GFAP,NfH,NfM,b-Tubulin
dose,,,,
0.2,0.01,0.01,0.01,0.01
20.0,0.01,0.01,0.01,0.01
2000.0,0.01,0.01,0.01,0.01
200000.0,0.01,0.10,0.28,0.01


### Estimated pg/ml with mixed targets: 2pg/ml NfL mixed with NfH, NfM, GFAP, b-Tubulin

In [7]:
xreact_mixed = pd.read_csv("iPLA_xreact_mixed.csv")
xreact_mixed.head(1)

,Unnamed: 0,well,position,target,sample_id,tube_id,rep_id,dilution,var_1,var_2,...,sybr (eds; multicomponent data),fam (eds; multicomponent data),cy5 (eds; multicomponent data),well (eds; melt curve raw data),well position (eds; melt curve raw data),reading (eds; melt curve raw data),temperature (eds; melt curve raw data),fluorescence (eds; melt curve raw data),derivative (eds; melt curve raw data),expt_folder_csv
0,4950,55,E7,NfL,NfL[1.9531]_progen#62008_NfH(50000)_progen#62210,NaN,28,1.0,1,0,...,"[152508.516, 154673.031, 157070.688, 157547.68...",NaN,NaN,55.0,E7,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[60.303, 60.795, 61.287, 61.779, 62.271, 62.76...","[1.879, 1.862, 1.84, 1.821, 1.799, 1.772, 1.75...","[0.044, 0.045, 0.045, 0.046, 0.046, 0.047, 0.0...",240514_e65_kruti


In [9]:
# Trim to means and relevant columns
xreact_mixed_means = xreact_mixed[['sample_id', 'expt_folder_csv', 'rep_id','SLR; log10(x); exc_std0; ct; mean_ng/L']].drop_duplicates(['expt_folder_csv', 'rep_id'])

### without CVs

In [10]:
# Identify correct column
value_col = 'SLR; log10(x); exc_std0; ct; mean_ng/L'

# Extract target (before '('), fix naming, and dose (inside '()')
xreact_mixed_means['target'] = xreact_mixed_means['sample_id'].apply(
    lambda x: "b-Tubulin" if x.startswith("bTubulin") else x.split("(")[0].strip()
)
xreact_mixed_means['dose'] = xreact_mixed_means['sample_id'].apply(
    lambda x: float(re.search(r"\(([^)]+)\)", x).group(1)) if re.search(r"\(([^)]+)\)", x) else np.nan
)

# Group by target and dose to compute mean
df_stats2 = (
    xreact_mixed_means.groupby(['dose', 'target'])[value_col]
    .mean()
    .reset_index()
)

# Pivot into a table: rows = dose, columns = target
df_result2 = df_stats2.pivot(index='dose', columns='target', values=value_col)

# Format to 2 decimal places
df_result2 = df_result2.round(2)

# Display or export
df_result2

target,NfL[1.9531]_progen#62008_GFAP,NfL[1.9531]_progen#62008_NfH,NfL[1.9531]_progen#62008_NfM,NfL[1.9531]_progen#62008_bTubulin
dose,,,,
5000.0,1.58,1.43,1.57,1.54
10000.0,1.54,1.60,1.62,1.59
50000.0,1.62,1.68,1.68,1.53


### with CVs

In [11]:
# Identify correct column
value_col = 'SLR; log10(x); exc_std0; ct; mean_ng/L'

# Extract target and dose from sample_id
xreact_mixed_means['target'] = xreact_mixed_means['sample_id'].apply(
    lambda x: "b-Tubulin" if x.startswith("bTubulin") else x.split("(")[0].strip()
)
xreact_mixed_means['dose'] = xreact_mixed_means['sample_id'].apply(
    lambda x: float(re.search(r"\(([^)]+)\)", x).group(1)) if re.search(r"\(([^)]+)\)", x) else np.nan
)

# Group by dose and target, compute mean and CV
grouped = xreact_mixed_means.groupby(['dose', 'target'])[value_col]
df_stats2 = grouped.agg(['mean', 'std']).reset_index()
df_stats2['cv_percent'] = (df_stats2['std'] / df_stats2['mean']) * 100

# Format mean(CV%) string
df_stats2['formatted'] = df_stats2.apply(lambda x: f"{x['mean']:.2f} ({x['cv_percent']:.0f}%)", axis=1)

# Pivot into final table
df_result2 = df_stats2.pivot(index='dose', columns='target', values='formatted')

# Save and Show table
df_result2.to_csv('CrossReactivity_mixed.csv')

df_result2

target,NfL[1.9531]_progen#62008_GFAP,NfL[1.9531]_progen#62008_NfH,NfL[1.9531]_progen#62008_NfM,NfL[1.9531]_progen#62008_bTubulin
dose,,,,
5000.0,1.58 (10%),1.43 (21%),1.57 (8%),1.54 (18%)
10000.0,1.54 (22%),1.60 (4%),1.62 (8%),1.59 (9%)
50000.0,1.62 (8%),1.68 (1%),1.68 (1%),1.53 (2%)
